<a href="https://colab.research.google.com/github/photominion777/exposure-value-to-light-value/blob/main/Light_Meter_Simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instruction:
1. Log into your Google account.
2. Press the "Run all" button at the top (the small ▶ triangle).
3. Choose the external lighting conditions with the lighting slider.
4. Adjust the aperture, shutter and ISO sliders.



In [3]:
#@title 🎛️ Interactive Digital Light Meter Simulator (Δ LV) { display-mode: "form" }

import math
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Global option lists representing the manufacturer display values
N_OPTIONS = [0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.5, 2.8, 3.2, 3.5, 4.0, 4.5, 5.0, 5.6, 6.3, 7.1, 8.0, 9.0, 10, 11, 13, 14, 16, 18, 20, 22, 25, 29, 32]
T_OPTIONS = ["1", "1/1.3", "1/1.6", "1/2", "1/2.5", "1/3.2", "1/4", "1/5", "1/6", "1/8", "1/10", "1/13", "1/15", "1/20", "1/25", "1/30", "1/40", "1/50", "1/60", "1/80", "1/100", "1/125", "1/160", "1/200", "1/250", "1/320", "1/400", "1/500", "1/640", "1/800", "1/1000", "1/1250", "1/1600", "1/2000", "1/2500", "1/3200", "1/4000", "1/5000", "1/6400", "1/8000"]
S_OPTIONS = [100, 125, 160, 200, 250, 320, 400, 500, 640, 800, 1000, 1250, 1600, 2000, 2500, 3200, 4000, 5000, 6400, 8000, 10000, 12800, 16000, 20000, 25600, 32000, 40000, 51200]

# Create an isolated output area to prevent function signature leaking
output_area = widgets.Output()

# Create a dedicated text label for the right side of the Lighting slider
lighting_text_right = widgets.Label(value="Sunny", layout=widgets.Layout(width='250px', margin='0 0 0 10px'))

def get_lighting_name(lv):
    # Definition of environmental light brackets
    ranges = [
        (19.5, 22.5, "Industrial Laser / Lab Light"),
        (18.5, 19.5, "Arc Welding / High-Power LED"),
        (17.0, 18.5, "Studio Flash / Searchlight"),
        (15.5, 17.0, "Extreme Sun (Snow/Sand)"),
        (14.5, 15.5, "Sunny"),
        (13.5, 14.5, "Hazy Sun"),
        (12.5, 13.5, "Bright Overcast"),
        (11.5, 12.5, "Overcast / Cloudy"),
        (10.5, 11.5, "Deep Shade"),
        (9.5, 10.5, "Sunset / Sunrise"),
        (8.5, 9.5, "Very Dynamic Twilight"),
        (7.5, 8.5, "Bright Street Lighting"),
        (6.5, 7.5, "Blue Hour / City Night"),
        (5.5, 6.5, "Bright Indoor"),
        (4.5, 5.5, "Standard Indoor"),
        (3.5, 4.5, "Living Room (Evening)"),
        (2.5, 3.5, "Dim Indoor"),
        (1.5, 2.5, "Distant Building Lights"),
        (0.5, 1.5, "Very Dim Interior"),
        (-0.5, 0.5, "Night Sky / City Skyline"),
        (-1.5, -0.5, "Dim Night Street"),
        (-2.5, -1.5, "Full Moon (Snow)"),
        (-3.5, -2.5, "Full Moon (Landscape)"),
        (-4.5, -3.5, "Quarter Moon"),
        (-5.5, -4.5, "Crescent Moon"),
        (-7.5, -5.5, "Starlight Night"),
    ]

    for low, high, name in ranges:
        if low <= lv < high:
            return name
    return "Transition / Mixed Light"

def calculate_and_display(L_input, N, t_str, S):
    # 1. MAP DISPLAY VALUES TO MATHEMATICALLY EXACT VALUES (1/3 STOPS)
    idx_N = N_OPTIONS.index(N)
    idx_t = T_OPTIONS.index(t_str)
    idx_S = S_OPTIONS.index(S)

    # Aperture (N): Base f/1.0 is at index 3. Steps scale by 2^(1/6)
    N_exact = 1.0 * (2**(1/6))**(idx_N - 3)

    # Shutter Speed (t): Base 1s is at index 0. Steps go to shorter times (2^(-1/3))
    t_exact = 1.0 * (2**(-1/3))**idx_t

    # ISO Speed (S): Base ISO 100 is at index 0. Steps scale by 2^(1/3)
    S_exact = 100.0 * (2**(1/3))**idx_S

    K = 12.5
    L = float(L_input)

    # SYSTEM SEPARATION: Left side (Camera) vs Right side (Environment)
    # Using exact values for APEX calculations to ensure perfect step alignment
    av = math.log2(N_exact**2)
    tv = math.log2(t_exact)
    sv = math.log2(S_exact / 100)
    LV_cam = av - tv - sv

    # Right side uses the physical luminance L
    LV_ext = math.log2(L * (100 / K))
    delta_lv = LV_ext - LV_cam

    # Dynamically update the custom text label on the right side of the slider
    current_name = get_lighting_name(LV_ext)
    lighting_text_right.value = current_name

    # EXACT COURIER ALIGNMENT MATRIX
    scale_header = " -5       -4       -3       -2       -1        0       +1       +2       +3       +4       +5  "

    scale_ticks = []
    for index in range(-15, 16):
        tick_value = index / 3.0

        if abs(delta_lv - tick_value) < (1.0 / 6.0) and -5.1 < delta_lv < 5.1:
            scale_ticks.append("▲")
        elif index % 3 == 0:
            scale_ticks.append("|")
        else:
            scale_ticks.append("·")

    prefix = "◀ " if delta_lv < -5.1 else "  "
    suffix = " ▶" if delta_lv > 5.1 else "  "

    scale_visual = prefix + "  ".join(scale_ticks) + suffix

    output_text = f"""
===================================================================================================
[ INTERNAL DIGITAL LIGHT METER ANALYSIS ]
===================================================================================================
Calculated Physical Luminance (L)  :  {L:.2f} cd/m²
Environmental Light Value (LV_ext) :  {LV_ext:.2f}
Camera Light Value (LV_cam)        :  {LV_cam:.2f}
---------------------------------------------------------------------------------------------------
Δ LV Deviation (Viewfinder Indicator) : {delta_lv:+.2f} stops
---------------------------------------------------------------------------------------------------
{scale_header}
{scale_visual}
===================================================================================================
"""

    if abs(delta_lv) < 0.17:
        output_text += "✅ EXPOSURE BALANCE: Perfectly matched. Light values are optimal."
    elif delta_lv > 0:
        output_text += "⚠️ OVEREXPOSURE: Camera configuration requires adjustment for a brighter scene."
    else:
        output_text += "⚠️ UNDEREXPOSURE: Camera configuration requires adjustment for a darker scene."

    # Add a sensor safety warning for extreme laboratory and industrial light intensities
    if LV_ext >= 18.5:
        output_text += "\n🔥 WARNING: High intensity light source. Prolonged exposure may damage the camera sensor."

    # Send output exclusively to our clean container area
    with output_area:
        clear_output(wait=True)
        display(HTML(f"<pre style='font-family: Courier New, Courier, monospace; line-height: 1.2; font-size: 14px;'>{output_text}</pre>"))

# MATHEMATICAL TRICK: Generate logarithmic third-stop options for physical Luminance (Extended to LV +22.0)
K_const = 12.5
luminance_options = []
for index in range(-18, 67):
    lv_val = index / 3.0
    l_nor = 2**lv_val
    l_phys = l_nor / (100 / K_const)
    if l_phys >= 10:
        l_phys = round(l_phys, 1)
    else:
        l_phys = round(l_phys, 2)
    if l_phys not in luminance_options:
        luminance_options.append(l_phys)

# Layout configuration for consistent and clean UI element dimensions
slider_layout = widgets.Layout(width='450px')
style = {'description_width': '120px'}

# Define control elements explicitly using global arrays
l_slider = widgets.SelectionSlider(options=luminance_options, value=4096.0, description="Lighting:", readout=False, layout=slider_layout, style=style)
n_slider = widgets.SelectionSlider(options=N_OPTIONS, value=16, description="Aperture (N):", layout=slider_layout, style=style)
t_slider = widgets.SelectionSlider(options=T_OPTIONS, value="1/100", description="Shutter (t):", layout=slider_layout, style=style)
s_slider = widgets.SelectionSlider(options=S_OPTIONS, value=100, description="ISO Speed (S):", layout=slider_layout, style=style)

# Wrap the main slider and the dynamic right-hand text label inside a horizontal box layout
lighting_row = widgets.HBox([l_slider, lighting_text_right])

# Link control updates directly to calculations
ui_controls = widgets.interactive_output(calculate_and_display, {'L_input': l_slider, 'N': n_slider, 't_str': t_slider, 'S': s_slider})

# Render the formatted interface component block
display(widgets.VBox([lighting_row, n_slider, t_slider, s_slider]), output_area)


Output()